# Python Modules, Packages, and Clean Architecture

**Session:** 2026-11-01 · OOP with Python  
**Module:** 01 · Python Fundamentals & Algorithmic Thinking

As computational software scales, keeping all definitions in a single script becomes unmaintainable. Python solves this via **Modules** and **Packages**:

1. **Module:** Any Python file (`.py`) containing statements, functions, classes, and constants. Modules provide a distinct namespace that prevents name collisions.
2. **Package:** A hierarchical directory structure containing modules and a special `__init__.py` file. Packages allow multi-level dot-notated namespaces (e.g., `package.subpackage.module`).

In this notebook, we explore the mechanics of Python's import system, directory structures, symbol exports (`__all__`), absolute vs relative imports, and apply these principles to construct a modular Audio Digital Signal Processing (DSP) pipeline based on `data/audio_pipeline_config.json`.

## Learning Outcomes

- Differentiate between scripts, modules, and packages in Python.
- Understand how Python resolves imports via `sys.path` and caches loaded modules in `sys.modules`.
- Use the `__init__.py` file to define package boundaries and control public exports with `__all__`.
- Contrast absolute imports with explicit relative imports (`.` and `..`).
- Build and execute a configurable DSP audio processing pipeline conforming to clean package design.

## 1. Modules as Namespaces and the Import System

When Python executes an `import module_name` statement:
1. Python checks `sys.modules` to see if the module has already been loaded. If present, it binds the cached module object to the current namespace.
2. If absent, Python searches each directory in `sys.path` (starting with the current working directory, followed by `PYTHONPATH`, standard library directories, and third-party site-packages).
3. The module code is executed top-to-bottom within its own isolated namespace dictionary (`module.__dict__`).
4. The entry is recorded in `sys.modules` for future lookups.

In [ ]:
import sys
from pathlib import Path

# Inspecting module resolution path:
print("sys.path entries (first 4):")
for i, p in enumerate(sys.path[:4]):
    print(f"  [{i}] {p}")

# Inspecting cached system modules:
print(f"\nTotal active modules loaded in sys.modules: {len(sys.modules)}")
print("Sample built-in modules in sys.modules:")
for mod in ["sys", "math", "json", "pathlib"]:
    if mod in sys.modules:
        print(f"  {mod:10s} -> {sys.modules[mod]}")

## 2. Structuring Modular Packages

Consider an application for audio signal processing. The code is logically grouped into file formats, digital filters, and spatial effects:

```text
audiokit/
├── __init__.py               # Package marker & public API facade
├── signal.py                 # Core AudioSignal representation
├── formats/                  # Subpackage for file parsing
│   ├── __init__.py
│   ├── wav.py
│   └── flac.py
├── filters/                  # Subpackage for frequency response
│   ├── __init__.py
│   ├── lowpass.py
│   └── equalizer.py
└── effects/                  # Subpackage for time/spatial effects
    ├── __init__.py
    ├── reverb.py
    └── spatializer.py
```

### The Role of `__init__.py`
1. **Package Marker:** Identifies the directory as a Python package to the import machinery.
2. **Package-Level Initialization:** Executes when the package is first imported.
3. **Public API Facade:** Selectively imports key classes from internal submodules, providing a clean import path for external users:
   ```python
   # Without facade:
   from audiokit.filters.lowpass import LowPassFilter
   # With facade in __init__.py:
   from audiokit import LowPassFilter
   ```

## 3. Controlling Symbol Exports with `__all__`

When a module or package defines `__all__ = ["SymbolA", "SymbolB"]`, it explicitly declares which symbols are exported when a client performs a wildcard import (`from module import *`). 

Restricting `__all__` prevents internal helper functions and unintended imports from leaking into the consumer's namespace.

In [ ]:
# Demonstrating __all__ export filtering mechanics:
class ExportSampleFilterA:
    pass


class ExportSampleFilterB:
    pass


def _internal_helper_fft():
    pass


# Explicitly define public exports:
__all__ = ["ExportSampleFilterA", "ExportSampleFilterB"]

print("Explicit public API exports (__all__):", __all__)

## 4. Relative vs Absolute Imports

Python supports two styles of imports within packages:

1. **Absolute Imports:** Specifies the full path from the project root:
   ```python
   from audiokit.filters.lowpass import LowPassFilter
   from audiokit.signal import AudioSignal
   ```
   *Advantage:* Explicit, unambiguous, and readable across large codebases.

2. **Explicit Relative Imports:** Uses leading dots to indicate the relative path from the current module's position:
   - `from . import signal` (same directory / subpackage)
   - `from .lowpass import LowPassFilter` (sibling module in same subpackage)
   - `from ..signal import AudioSignal` (parent package module)
   *Advantage:* Packages can be renamed or relocated without rewriting internal imports.

## 5. Case Study: Executing the Modular DSP Pipeline

We now implement the modular audio processing architecture modeled in `data/audio_pipeline_config.json`.
We build:
1. `AudioSignal`: A container encapsulating raw audio samples, sample rate, and channels.
2. `BaseDSPModule`: An abstract base class for signal processing units.
3. Concrete DSP filters: `LowPassFilter`, `ParametricEqualizer`, and `ConvolutionReverb`.
4. `AudioProcessingPipeline`: A container that ingests the JSON config, instantiates active DSP modules, and executes the signal processing chain.

In [ ]:
import json
import math
from abc import ABC, abstractmethod
from dataclasses import dataclass


@dataclass
class AudioSignal:
    """Container for multi-channel digital audio signals."""

    samples: list[float]
    sample_rate_hz: int
    channels: int
    title: str = "Synthesized Test Signal"

    @property
    def duration_seconds(self) -> float:
        return len(self.samples) / (self.sample_rate_hz * self.channels)

    @property
    def peak_amplitude(self) -> float:
        return max(abs(s) for s in self.samples) if self.samples else 0.0


class BaseDSPModule(ABC):
    """Abstract base class for all digital signal processing filters."""

    def __init__(self, name: str, params: dict) -> None:
        self.name = name
        self.params = params

    @abstractmethod
    def process(self, signal: AudioSignal) -> AudioSignal:
        """Transform the input audio signal."""
        pass


class LowPassFilter(BaseDSPModule):
    """Simulated single-pole low-pass filter attenuating high frequencies."""

    def process(self, signal: AudioSignal) -> AudioSignal:
        cutoff = self.params.get("cutoff_frequency_hz", 8000)
        # Smoothing alpha parameter based on cutoff frequency
        dt = 1.0 / signal.sample_rate_hz
        rc = 1.0 / (2 * math.pi * cutoff)
        alpha = dt / (rc + dt)

        filtered = []
        prev = 0.0
        for s in signal.samples:
            current = prev + alpha * (s - prev)
            filtered.append(current)
            prev = current

        return AudioSignal(
            samples=filtered,
            sample_rate_hz=signal.sample_rate_hz,
            channels=signal.channels,
            title=f"{signal.title} + [LowPassFilter cutoff={cutoff}Hz]",
        )


class ParametricEqualizer(BaseDSPModule):
    """Simulated multi-band parametric equalizer applying gain stages."""

    def process(self, signal: AudioSignal) -> AudioSignal:
        bands = self.params.get("bands", [])
        # Scale amplitude based on aggregate band gain
        total_gain_db = sum(b.get("gain_db", 0.0) for b in bands)
        linear_gain = 10.0 ** (total_gain_db / 20.0)

        # Apply scaled gain to samples with soft limiting
        shaped = [math.tanh(s * linear_gain) for s in signal.samples]

        return AudioSignal(
            samples=shaped,
            sample_rate_hz=signal.sample_rate_hz,
            channels=signal.channels,
            title=f"{signal.title} + [Equalizer bands={len(bands)}]",
        )


class ConvolutionReverb(BaseDSPModule):
    """Simulated spatial reverb effect blending wet decay reflections."""

    def process(self, signal: AudioSignal) -> AudioSignal:
        wet_mix = self.params.get("wet_dry_mix", 0.35)
        decay = self.params.get("decay_time_seconds", 1.8)

        reverbed = []
        for i, s in enumerate(signal.samples):
            # Simulated exponential decay echo
            echo = 0.2 * s * math.exp(-i / (signal.sample_rate_hz * 0.1))
            blended = (1.0 - wet_mix) * s + wet_mix * echo
            reverbed.append(blended)

        return AudioSignal(
            samples=reverbed,
            sample_rate_hz=signal.sample_rate_hz,
            channels=signal.channels,
            title=f"{signal.title} + [Reverb decay={decay}s mix={wet_mix:.2f}]",
        )

In [ ]:
# Loading DSP Pipeline configuration
config_file = Path("../data/audio_pipeline_config.json")
if not config_file.exists():
    config_file = Path("data/audio_pipeline_config.json")

with config_file.open("r", encoding="utf-8") as f:
    pipeline_config = json.load(f)

# Module registry mapping config class names to concrete classes
MODULE_REGISTRY = {
    "LowPassFilter": LowPassFilter,
    "ParametricEqualizer": ParametricEqualizer,
    "ConvolutionReverb": ConvolutionReverb,
}

# Instantiate active pipeline stages
pipeline_stages = []
for mod in pipeline_config["dsp_modules"]:
    if not mod["enabled"]:
        continue
    cname = mod["class_name"]
    if cname in MODULE_REGISTRY:
        stage = MODULE_REGISTRY[cname](name=cname, params=mod["parameters"])
        pipeline_stages.append(stage)

print(f"Loaded Pipeline: {pipeline_config['pipeline_title']} (v{pipeline_config['version']})")
print(f"Active Stages Count: {len(pipeline_stages)}")
for idx, st in enumerate(pipeline_stages, 1):
    print(f"  Stage {idx}: {st.name}")

In [ ]:
# Generate synthetic test audio signal: 1 second sine wave at 440 Hz (A4)
sample_rate = pipeline_config["global_audio_specs"]["sample_rate_hz"]
freq_hz = 440.0
total_samples = sample_rate  # 1 second of audio

raw_audio = [math.sin(2 * math.pi * freq_hz * (t / sample_rate)) for t in range(total_samples)]
current_signal = AudioSignal(
    samples=raw_audio, sample_rate_hz=sample_rate, channels=1, title="440Hz Sine Tone"
)

print(f"Initial Signal: {current_signal.title}")
print(f"  Sample Rate: {current_signal.sample_rate_hz} Hz | Samples: {len(current_signal.samples)}")
print(
    f"  Duration: {current_signal.duration_seconds:.2f}s | Peak: {current_signal.peak_amplitude:.4f}\n"
)

# Process signal sequentially through the DSP pipeline
for stage in pipeline_stages:
    print(f"Executing DSP stage: {stage.name}...")
    current_signal = stage.process(current_signal)

print("\nPipeline execution complete!")
print(f"Final Signal State: {current_signal.title}")
print(f"Final Peak Amplitude: {current_signal.peak_amplitude:.4f}")